# Linear Regression

<img src="./img/4-Linear-Regression-Assumptions.webp" width="500px">
<br><br>

<span style="font-size: 70%">For explanation of <b>homoscedasticity</b> refer to <a href="https://en.wikipedia.org/wiki/Homoscedasticity_and_heteroscedasticity">Wikipedia</a>. 
A good video on homoscedasticity can be found <a href="https://www.youtube.com/watch?v=LVvu9UgNhTU">here</a> (explanation from 5:10').</span>

### Model

we assume that:

> &nbsp;  
> $y_i = \theta_0 + \theta_1x_i + \epsilon_i$
> <br><br>

for each $x_i$ (each row in our data).

<span style="font-size: 70%"><b>Note:</b> As opposed to the book "Data Science from Scratch", we use <i>&theta;<sub>i</sub></i> as parameters, for <i>&alpha;</i> and <i>&beta;</i>. This conforms to Stanford nomenclature.</span>
<br><br>

In Python we can express this as:

In [ ]:
# some imports  and definitions first
import matplotlib.pyplot as plt
from typing import Tuple, List
Vector = List[float]
import numpy as np
import seaborn as sns                               # seaborn provides combined graphs

In [ ]:
# we can predict a target value y_i as:
def predict(theta_0: float, theta_1: float, x_i: float) -> float:
    return theta_1 * x_i + theta_0

# and calculate the error for a given y_i
def error(theta_0: float, theta_1: float, x_i: float, y_i: float) -> float:
    """
    The error from predicting beta * x_i + alpha
    when the actual value is y_i
    """
    return predict(theta_0, theta_1, x_i) - y_i

# errors might get cancelled out, thus use squared errors
def sum_of_sqerrors(theta_0: float, theta_1: float, x: Vector, y: Vector) -> float:
    return sum(error(theta_0, theta_1, x_i, y_i) ** 2
               for x_i, y_i in zip(x, y))

Our goal is to minimize the error $\epsilon_i$.

In [ ]:
# refer to 3b Statistics for details on correlation, standard deviation and their substitute with numpy
#   correlation(x, y) ... np.corrcoef(x, y)[0][1]
#   mean(x) ... np.mean(x)
#   standard_deviation(x) ... np.std(x)

def least_squares_fit(x: Vector, y: Vector) -> Tuple[float, float]:
    """
    Given two vectors x and y,
    find the least-squares values of alpha and beta
    """
    theta_1 = np.corrcoef(x, y)[0][1] * np.std(y) / np.std(x)
    theta_0 = np.mean(y) - theta_1 * np.mean(x)
    return theta_0, theta_1


x = [i for i in range(-100, 110, 10)]
y = [3 * i - 5 for i in x]

# Should find that y = 3x - 5
assert least_squares_fit(x, y) == (-5, 3)

We can apply this to our friends dataset.

In [ ]:
# load Friends dataset

friends = np.loadtxt('data/Friends.csv', skiprows=1, delimiter=',')
num_friends_good = friends[:,0]
daily_minutes_good = friends[:,1]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10,5))
ax[0].plot(num_friends_good)
ax[0].set_title('Number of friends')
ax[1].hist(daily_minutes_good)
ax[1].set_title('Minutes per day online')
plt.show()

### Fitting the model

In [ ]:
# find the least squared fit
theta_0, theta_1 = least_squares_fit(num_friends_good, daily_minutes_good)

print(f"LSF parameters: theta_0: {theta_0:2.2f}, theta_1: {theta_1:2.3f}")

### Interpretation

Our offset is $\theta_0$ with a value of about _22.95_.

> &nbsp;  
> $\rightarrow$ On average, each user spends a __minimum of 23 minutes per day__ online (on average).
> <br><br>

$\theta_1$ is _0.904_.

> &nbsp;  
> $\rightarrow$ __Each friend__ connected __adds a minute__ to the time spent online.
> <br><br>

Let's observe this visually.

In [ ]:
# visualize the data and best fit regression line

plt.scatter(num_friends_good, daily_minutes_good)
plt.plot([0, 50], [theta_0, theta_0 + 50 * theta_1], c='red')
plt.show()

<span style="font-size: 70%">Figure 1: Friends data is estimated by a regression line (red).</span>
<br><br>

Is this accurate or representative?  
How does this claim work with larger number of friends?  

Assume a model that represents:

$\theta_0 = mean(y)$, $\theta_1 = 0$ ... in fact the constant model

In [ ]:
plt.scatter(num_friends_good, daily_minutes_good)
plt.plot([0, 50], [np.mean(daily_minutes_good),
         np.mean(daily_minutes_good)], c='red', linestyle='dashed')
plt.show()

<span style="font-size: 70%">Figure 2: Friends data estimated by the mean of y.</span>
<br><br>



<table>
<tr>
<td style="border-style: none"><img src="./img/0_reference.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><u>Further reading:</u>
<ul>
<li>Chapter 14 (p 239ff) from "Data Science from Scratch"</li>
<li>Scikit Learn on <a href="https://scikit-learn.org/stable/modules/model_evaluation.html#r2-score-the-coefficient-of-determination">R2 score</a></li>
</ul>
</td>
</tr>
</table>

### Model Metrics: $R^2$ (Coefficient of determination)

The __Least Squared Error__ model must be at least as good as the constant model with $\theta_0 = mean(y)$ and $\theta_1 = 0$.

> &nbsp;  
> $R^2\left (y_i, \widehat{y}_i \right) = 1 - \frac{\sum_{i=1}^{n}\left( y_i - \widehat{y}_i \right)^2}{\sum_{i=1}^{n}\left( y_i - \overline{y} \right)^2}$
> <br><br>

with:

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\widehat{y}_i$ ... predicted i-th value of $y$ and  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $y_i$ ... true i-th value of $y$  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\overline{y} = \frac{1}{n}\sum_{i=1}^{n} y_i$ and  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; $\sum_{i=1}^{n}\left( y_i - \widehat{y}_i \right)^2 = \sum_{i=1}^{n} \epsilon_i^2$

<br><br>
The $R^2$ score represents the proportion of variance (of $y$) that has been explained by the independent variable(s) in the model.  
It indicates goodness of fit and therefore measures how well unseen samples are likely to be predicted by the model.

As such, $R^2$ is a __measurement for the quality__ of the results. Metrics play a significant role in evaluating model quality.

In [ ]:
# let's put this into python code

def total_sum_of_squares(y: Vector) -> float:
    """the total squared variation of y_i's from their mean"""
    return sum(v ** 2 for v in y - np.mean(y))


def r_squared(theta_0: float, theta_1: float, x: Vector, y: Vector) -> float:
    """
    the fraction of variation in y captured by the model, which equals
    1 - the fraction of variation in y not captured by the model
    """
    return 1.0 - (sum_of_sqerrors(theta_0, theta_1, x, y) /
                  total_sum_of_squares(y))


rsq = r_squared(theta_0, theta_1, num_friends_good, daily_minutes_good)
assert 0.328 < rsq < 0.330


$R^2$ score depends on the dataset. It's value is __not neccessarily comparable over different datasets__.

In [ ]:
# R2 for our model
rsq_lr = r_squared(theta_0, theta_1, num_friends_good, daily_minutes_good)

# R2 for the constant model
rsq_cm = r_squared(np.mean(daily_minutes_good), 0, num_friends_good, daily_minutes_good)

print(f"R2 for the linear model:  {rsq_lr:.3f}\nR2 for the constant model: {rsq_cm:.3f}")

### Interpretation

$0 \le R^2 \le 1$

> &nbsp;  
> $\rightarrow$ the constant model has an $R^2$ score of 0. The target variable __cannot be explained by the__ (values of the) __independent variable__ at all. <br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; Our __linear model__ has an $R^2$ score of 0.329, indicating a __better fit__ (1 being perfect).
> <br><br>



<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>Discuss: Are we satisfied with the model above? What can we improve?</td>
</tr>
</table>

### Using Gradient Descent

So far we have used correlation and linear algebra to define, calculate and evaluate our model.

In "3c Gradient Descent" we introduced a general method to approximate parameters in order to optimize model fit.

Let's apply this knowledge here.

In [ ]:
import random
import tqdm


def gradient_step(v: Vector, gradient: Vector, step_size: float) -> Vector:
    """Moves `step_size` in the `gradient` direction from `v`"""
    assert len(v) == len(gradient)
    step = [step_size * v_i for v_i in gradient]
    return [v_i + w_i for v_i, w_i in zip(v, step)]


# initial value: num_epochs = 10000
num_epochs = 10000                                      # change from 1000 ... 10000 in steps of 1000
random.seed(0)

guess = [random.random(), random.random()]              # choose random value to start
# initial value: learning_rate = 0.00001
learning_rate = 0.00001                                 # change from 0.00001 to 0.001 in steps of 3 and 5
loss = 2^31

with tqdm.trange(num_epochs) as t:
    for _ in t:
        theta_0, theta_1 = guess

        # Partial derivative of loss with respect to theta_0
        grad_0 = sum(2 * error(theta_0, theta_1, x_i, y_i)
                        for x_i, y_i in zip(num_friends_good,
                                            daily_minutes_good))

        # Partial derivative of loss with respect to theta_1
        grad_1 = sum(2 * error(theta_0, theta_1, x_i, y_i) * x_i
                        for x_i, y_i in zip(num_friends_good,
                                            daily_minutes_good))

        # Compute loss to stick in the tqdm description
        loss = sum_of_sqerrors(theta_0, theta_1,
                                num_friends_good, daily_minutes_good)
        t.set_description(f"loss: {loss:.3f}, theta_0: {theta_0:2.2f}, theta_1: {theta_1:2.3f}")

        # Finally, update the guess
        guess = gradient_step(guess, [grad_0, grad_1], -learning_rate)

    t.close()

# We should get pretty much the same results:
theta_0, theta_1 = guess
print(f"Gradient descent parameters: theta_0: {theta_0:.2f}, theta_1: {theta_1:.3f}")
#assert 22.9 < theta_0 < 23.0
#assert 0.9 < theta_1 < 0.905


<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>
1. Change the number of epoches (1000 ... 10000) and verify that the results are correct.<br>&nbsp;&nbsp;&nbsp; What is a good value for epoches?<br>&nbsp;&nbsp;&nbsp; What influences the choice?<br>
2. Change the learning rate (0.00001, 0.00003, 0.00005, 0.0001, ...).<br>&nbsp;&nbsp;&nbsp; What is a good compromise for value?<br>&nbsp;&nbsp;&nbsp; When does the model stop to converge?</td>
</tr>
</table>

# Hyper parameters

<img src="./img/4_model-parameter-vs-hyperparameter3.png" width="500px">
<br><br>

In our model we used ___parameters___ $\theta_0$ and $\theta_1$ to describe the variable part of the model. We found sufficient values through __optimization__.

___`Parameters`___ are intrinsic to the model and influence the __result of the model__.
<br><br>  

In the example above we used $epoches$ and $learning\underline{\space\space}rate$. 

These parameters are external to the model calculation. They influence the performance or convergence of the calculation.

Therefore they are called ___`Hyper Parameters`___ to the model and are mainly used to __tune__ the performance of __models__.
<br><br>

<img src="./img/4_hyperparameter_tuning.jpeg" width="700px">
<br><br>

Tuning of hyperparameters happens after primary training of the model completed.
<br><br>

# Introducing Scikit Learn

Writing code to analyze data in Python can be instructive. Maintaining code is time consuming and distracts from the task at hand - data analytics.

A more efficient way would be to use an established set of libraries that are well maintained, cover a wide range of methods and provide tools to support data analysis.

Such a library is `Scikit Learn`. It is founded on the scientific stack (Numpy, SciPy, Pandas, matplotlib, etc.), maintained as open source by a focused community.

<img src="./img/4_ml_map.png" width="700px">
<br><br>

After having introduced the foundation of a topic, we will use `scikit-learn` to provide optimized models.

<table>
<tr>
<td style="border-style: none"><img src="./img/0_reference.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><u>Further reading:</u>
<ul>
<li>Get acquainted to <a href="https://scikit-learn.org/">scikit-learn</a></li>
</ul>
</td>
</tr>
</table>

### Linear Regression with `scikit-learn`

We've manually coded Linear Regression in Python. While not teriffically hard, the code gets cluttered and slow on large datasets.

Let's rewrite using `scikit-learn`.

In [ ]:
# some required imports
# if not installed, pip install sklearn

from sklearn import linear_model
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# prepare the data
# X contains the vectors of independent variables
# y is the target variable

X = num_friends_good.reshape(-1,1)
y = daily_minutes_good.reshape(-1,1)

# create an estimator model and fit it to the data
lr = linear_model.LinearRegression()
lr.fit(X, y)

print(f"LR intercept: {lr.intercept_[0]:.3f}, coefficient: {lr.coef_[0][0]:.3f}")

In [ ]:
# we use predict to calculate the predicted values

y_pred = lr.predict(np.array([0, 50]).reshape(-1,1))
y_pred

In [ ]:
# visualize the data and predicted regression line

plt.scatter(num_friends_good, daily_minutes_good)
plt.plot([0, 50], y_pred, c='red')
plt.show()

<span style="font-size: 70%">Figure 3: Friends data is estimated by a regression line (red) generated using scikit-learn. The result is similar to the one on Figure 1.</span>
<br><br>

### Metrics

In [ ]:
# test the quality of our estimate

print(f"Mean Squared Error: {mean_squared_error(X, y):.3f}")
print(f"R2 score:         {r2_score(y, lr.predict(X)):.3f}")
print(f"Regression score: {lr.score(X, y):.3f}")

In [ ]:
# analyze visually
sns.set_theme(style="darkgrid")
sns.jointplot(x=num_friends_good, y=daily_minutes_good, kind='reg', color='b')
plt.show()

In [ ]:
sns.residplot(x=num_friends_good, y=daily_minutes_good)
plt.show()

<table>
<tr>
<td style="border-style: none"><img src="./img/0_students_input.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><h5>Students task:</h5>Discuss: How do these graphs align with the 5 axioms for LR at the top of this notebook?</td>
</tr>
</table>

<table>
<tr>
<td style="border-style: none"><img src="./img/0_reference.png" height="100px"></td>
<td style="border-style: none">&nbsp;&nbsp;</td>
<td style="border-style: none; vertical-align: middle"><u>Further reading:</u>
<ul>
<li><a href="https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares">Ordinary Least Squares Linear Regression</a></li>
<li><a href="https://seaborn.pydata.org/examples/regression_marginals.html">Using seaborn for combined visualization</a></li>
</ul>
</td>
</tr>
</table>

`Scikit-Learn` not only provides a wide range of methods for data analysis but provides extensive documentation and example code.

`seaborn` provides enhanced visuals on top of `matplotlib`

<span style="font-size: 128px">&#9749;</span> Coffee break!